- self：关联实例对象本身
- super：关联父类（超类）的方法调用

# 一、self
1. 核心定义：
- self 是 Python 类的实例方法中第一个参数的约定俗成的名称（不是关键字，只是大家都遵守的规范），它指向当前类的实例对象本身。
- 当你调用实例方法时，Python 会自动将实例对象作为第一个参数传递给 self，你无需手动传入；在类内部，通过 self 可以访问该实例的属性和方法。
2. 核心作用：
- 访问 / 修改实例属性：类内部通过 self.属性名 操作实例的属性（区别于局部变量）
- 调用实例方法：类内部通过 self.方法名() 调用当前实例的其他方法
- 区分实例属性和局部变量：避免同名的局部变量覆盖实例属性
- 标识实例唯一性：每个实例的 self 指向自身，保证不同实例的属性相互独立
3. 示例
- 示例1:self访问实例属性

In [1]:
class Person:
    def __init__(self, name, age):
        # self.name 是实例属性，绑定到当前实例
        self.name = name
        self.age = age

    def introduce(self):
        # 通过 self 访问实例属性
        print(f"我叫{self.name}，今年{self.age}岁")

# 创建实例
p1 = Person("张三", 20)
p2 = Person("李四", 25)

# 调用方法时，Python自动把p1/p2传给self
p1.introduce()  # 我叫张三，今年20岁
p2.introduce()  # 我叫李四，今年25岁

# 直接通过实例访问属性（本质是self绑定的结果）
print(p1.name)  # 张三

我叫张三，今年20岁
我叫李四，今年25岁
张三


- 示例2:self区分实例属性和局部变量

In [2]:
class Student:
    def __init__(self, score):
        # 实例属性：self.score
        self.score = score

    def update_score(self, score):
        # 局部变量：score（方法参数）
        # 实例属性：self.score（通过self区分）
        if score >= 0 and score <= 100:
            self.score = score  # 修改实例属性
            print(f"分数更新为：{self.score}")
        else:
            print("分数不合法")

s = Student(80)
s.update_score(90)  # 分数更新为：90
print(s.score)      # 90（实例属性被修改）

分数更新为：90
90


- 示例3:self调用实例的其他方法

In [3]:
class Calculator:
    def __init__(self, a, b):
        self.a = a
        self.b = b

    def add(self):
        return self.a + self.b

    def print_result(self):
        # 通过 self 调用当前实例的 add 方法
        result = self.add()
        print(f"{self.a} + {self.b} = {result}")

calc = Calculator(3, 5)
calc.print_result()  # 3 + 5 = 8

3 + 5 = 8


4. 常见误区
- 误区1:self 是关键字 → 错误！self 只是约定俗成的名称，你可以换成 this/me 等，但强烈不推荐（违反 PEP8 规范，代码可读性差）：

In [ ]:
class Test:
    def __init__(this, name):  # 用this代替self，语法合法但不推荐
        this.name = name
    def say(this):
        print(this.name)
t = Test("测试")
t.say()  # 测试

- 误区2:类方法 / 静态方法需要 self → 错误！self 只用于实例方法，类方法用 cls（指向类本身），静态方法不需要第一个特殊参数：

In [4]:
class MyClass:
    # 实例方法：需要self
    def instance_method(self):
        print("这是实例方法")

    # 类方法：需要cls
    @classmethod
    def class_method(cls):
        print("这是类方法")

    # 静态方法：不需要self/cls
    @staticmethod
    def static_method():
        print("这是静态方法")

# 调用方式
obj = MyClass()
obj.instance_method()  # 实例调用
MyClass.class_method() # 类调用
MyClass.static_method()# 类调用

这是实例方法
这是类方法
这是静态方法


- 误区3:手动传递 self 参数 → 错误！调用实例方法时，Python 会自动传递，手动传会报错：

In [ ]:
p = Person("张三", 20)
# p.introduce(p)  # 报错：TypeError: introduce() takes 1 positional argument but 2 were given

# super()关键字
1. 核心定义：super() 是 Python 内置函数，用于调用父类（超类）的方法，尤其是在子类继承父类的场景中。它的核心价值是：
    - 无需手动指定父类名称，代码更灵活（比如修改父类名时，无需修改子类的调用逻辑）；
    - 自动遵循MRO（方法解析顺序），处理多继承时避免混乱。
2. 核心作用：
    - 初始化父类属性：子类 __init__ 中调用父类 __init__，复用父类的属性初始化逻辑
    - 调用父类被重写的方法：子类重写父类方法后，仍能调用父类原方法（扩展而非完全替换）
    - 处理多继承的方法调用：按 MRO 顺序自动查找父类方法，避免手动指定的错误
3. 基础语法：
    - super().方法名(参数)：Python3 单继承 / 多继承，简化写法，自动绑定当前子类和实例；
    - super(子类名, self).方法名(参数)：兼容 Python2 / 显式指定，完整写法，效果与简化版一致。
4. 代码示例：
- 示例1:但继承中用super()初始化父类属性：这是super最常用的场景，避免子类重复编写父类的初始化逻辑。

In [5]:
# 父类：Person
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        print(f"我叫{self.name}，今年{self.age}岁")

# 子类：Student（继承Person）
class Student(Person):
    def __init__(self, name, age, student_id):
        # 调用父类的__init__，初始化name和age
        super().__init__(name, age)
        # 子类扩展的属性
        self.student_id = student_id

    def introduce(self):
        # 先调用父类的introduce方法
        super().introduce()
        # 再扩展子类的逻辑
        print(f"我的学号是{self.student_id}")

# 测试
s = Student("王五", 18, "2024001")
s.introduce()
# 输出：
# 我叫王五，今年18岁
# 我的学号是2024001

我叫王五，今年18岁
我的学号是2024001


- 示例2:子类重写父类方法后，用super()调用父类原方法：如果子类完全重写父类方法，又想保留父类的逻辑，可以通过super()调用

In [6]:
class Animal:
    def make_sound(self):
        print("动物发出声音")

class Dog(Animal):
    def make_sound(self):
        # 先调用父类的make_sound
        super().make_sound()
        # 再添加子类的逻辑
        print("汪汪汪！")

dog = Dog()
dog.make_sound()
# 输出：
# 动物发出声音
# 汪汪汪！

动物发出声音
汪汪汪！


- 示例3:多继承中super()遵循MRO顺序：多继承时，Python 会按 MRO（Method Resolution Order） 顺序查找父类方法，super() 会自动遵循这个顺序，避免手动指定的混乱。

In [7]:
# 父类1：A
class A:
    def show(self):
        print("A的show方法")

# 父类2：B（继承A）
class B(A):
    def show(self):
        super().show()
        print("B的show方法")

# 父类3：C（继承A）
class C(A):
    def show(self):
        super().show()
        print("C的show方法")

# 子类：D（继承B、C）
class D(B, C):
    def show(self):
        super().show()
        print("D的show方法")

# 查看D的MRO顺序
print(D.__mro__)
# 输出：(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)

# 调用D的show方法，super()按MRO顺序执行
d = D()
d.show()
# 输出：
# A的show方法
# C的show方法
# B的show方法
# D的show方法

(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)
A的show方法
C的show方法
B的show方法
D的show方法


5. 常见误区
- 误区 1：super() 只能调用直接父类 → 错误！super() 按 MRO 顺序查找，可能调用间接父类（如示例 3 中，B 的 super () 调用了 C 的方法，而非直接父类 A）；
- 误区 2：super().__init__() 必须传 self → 错误！super() 会自动传递实例引用，只需传父类 __init__ 所需的其他参数；
- 误区 3：多继承时必须用 super() → 不是必须，但强烈推荐！如果手动指定父类（如 Person.__init__(self, name, age)），多继承时容易导致父类方法被多次调用，而 super() 会保证每个父类方法只执行一次。